In [66]:
import pandas as pd
import numpy as np
import re
import unicodedata
from pathlib import Path

In [67]:
# Uses the same normalization done to lookup file's 'Match key' column.
# Rule identical to that used by Claude to generate tourism data sheets.
def match_key(s):
    s = unicodedata.normalize('NFKD', str(s))
    s = ''.join(c for c in s if not unicodedata.combining(c))
    s = re.sub(r"[´’'`]", '', s.lower())
    return re.sub(r'[^a-z0-9]+', ' ', s).strip()

In [68]:
# Read ISO3 code sheet and gather the essential columns
# Using a version 'improved' by Claude
iso3_sheet = pd.read_excel("data/country_name_iso3_lookup.xlsx", sheet_name="Name Lookup")
iso3_sheet['key'] = iso3_sheet['Match key'].map(match_key)
iso3_codes = iso3_sheet[['key', 'ISO3']].drop_duplicates()

conflicts = iso3_codes.groupby('key')['ISO3'].nunique()
assert (conflicts <= 1).all(), conflicts[conflicts > 1]
iso3_codes = iso3_codes.drop_duplicates('key')

subnational = set(iso3_sheet.loc[iso3_sheet['Kind'] == 'sub-national', 'key'])

print(iso3_codes.head())

                    key ISO3
0           afghanistan  AFG
1                angola  AGO
2               albania  ALB
3               andorra  AND
4  united arab emirates  ARE


In [69]:
# Collect all tourism data
tourism_folder = Path("data/yearbook_tourism")
tourism_data = []
for file_path in tourism_folder.iterdir():
    if file_path.is_file():
        print(f"Opening {file_path.name}")
        data_file = pd.read_excel(file_path, sheet_name="Tourist Arrivals")
        data_file['Edition'] = file_path.name
        tourism_data.append(data_file)

Opening tourist_arrivals_by_origin_2000-2004.xlsx
Opening tourist_arrivals_by_origin_2004-2008.xlsx
Opening tourist_arrivals_by_origin_2008-2012.xlsx
Opening tourist_arrivals_by_origin_2012-2016.xlsx
Opening tourist_arrivals_by_origin_2016-2020.xlsx


In [78]:
# Combine all tourism data to a single dataframe and add ISO3 codes.
total_tourism_data = pd.concat(tourism_data)
total_tourism_data['Destination'] = total_tourism_data['Destination'].map(match_key)
total_tourism_data['Origin'] = total_tourism_data['Origin'].map(match_key)

unknown = (set(total_tourism_data['Destination']) | set(total_tourism_data['Origin'])) - set(iso3_codes['key'])
assert not unknown, unknown

total_tourism_data = total_tourism_data[~total_tourism_data['Destination'].isin(subnational) & ~total_tourism_data['Origin'].isin(subnational)]

total_tourism_data = pd.merge(total_tourism_data, iso3_codes[['key', 'ISO3']],
                              left_on='Destination',
                              right_on='key',
                              how='left').drop(columns=['key'])
total_tourism_data = total_tourism_data.rename(columns={"ISO3": "destinationISO"})
total_tourism_data = pd.merge(total_tourism_data, iso3_codes[['key', 'ISO3']],
                              left_on='Origin',
                              right_on='key',
                              how='left').drop(columns=['key'])
total_tourism_data = total_tourism_data.rename(columns={"ISO3": "originISO"})

print(len(total_tourism_data))

78497


In [80]:
# Check for rows with NaN in destination or origin ISO codes.
nan_destinations = total_tourism_data[total_tourism_data['destinationISO'].isna()]
nan_origins = total_tourism_data[total_tourism_data['originISO'].isna()]
print(len(nan_destinations), "destinations without ISO codes")
print(len(nan_origins), 'origins without ISO codes')

37 destinations without ISO codes
5956 origins without ISO codes


In [72]:
# Combine rows by origin/destination so that all correct year data is on the same row. Additionally reorganizes columns.

# This chunk of code is Claude generated to additionally account for conflicting data siding with more recent editions and dealing with conflicts caused by country subregions.
total_tourism_data['Series'] = total_tourism_data['Source table'].astype(str) + "|" + total_tourism_data['Basis']
keys = ['destinationISO', 'originISO']
years = [c for c in total_tourism_data.columns if str(c).isdigit()]
labels = ['Destination', 'Origin', 'Region', 'Subregion', 'Measure', 'Basis', 'Source table', 'Note', 'Series']
total_tourism_data = total_tourism_data.groupby(keys + ["Edition"], as_index = False).agg({**{c: 'first' for c in labels}, **{y: (lambda s: s.sum(min_count=1)) for y in years}})
varies = (total_tourism_data.groupby(keys)['Series'].nunique().gt(1).rename('measure_varies').reset_index())
merged_tourism_data = (total_tourism_data.sort_values('Edition').groupby(keys, as_index=False).last().drop(columns=['Edition', 'Series']).merge(varies, on=keys))

front_columns = ['destinationISO', 'originISO', 'Destination', 'Origin', 'Region', 'Subregion', 'Measure', 'Basis', 'Source table', 'Note']
year_columns = [col for col in merged_tourism_data.columns if col not in front_columns]
merged_tourism_data = merged_tourism_data[front_columns + year_columns]
print(merged_tourism_data.columns)
print(len(merged_tourism_data))

Index(['destinationISO', 'originISO', 'Destination', 'Origin', 'Region',
       'Subregion', 'Measure', 'Basis', 'Source table', 'Note', '2000', '2001',
       '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010',
       '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019',
       '2020', 'measure_varies'],
      dtype='str')
21105


In [73]:
# Check for rows with duplicates
row_duplicates = merged_tourism_data[merged_tourism_data.duplicated(subset=['destinationISO', 'originISO'], keep=False)]
print(len(row_duplicates))

0


In [74]:
# Check for rows with NaN in destination or origin ISO codes.
nan_destinations = merged_tourism_data[merged_tourism_data['destinationISO'].isna()]
nan_origins = merged_tourism_data[merged_tourism_data['originISO'].isna()]
print(len(nan_destinations))
print(len(nan_origins))

0
0


In [75]:
merged_tourism_data.to_csv('processed_data/processed_tourism.csv', index=False)